In [1]:
from pathlib import Path
from typing import Dict, Any

In [18]:
import subprocess
import sys
from tqdm.notebook import tqdm
import spacy
from typing import Dict, Any, Optional, Union
import json
import pandas as pd 
from refined.inference.processor import Refined, Span
import difflib

class PipelineInformationExtractorRefined:
    def __init__(self):
        self.model = Refined.from_pretrained(
            model_name="wikipedia_model_with_numbers", entity_set="wikipedia"
        )
    def _create_span(self, span: Span, index: int):
        entity =span.predicted_entity 
        features_object = (
            {
                "linking": {
                    "title": entity.wikipedia_entity_title,
                    "top_candidate": {
                        "url": "https://www.wikidata.org/wiki/"
                        + entity.wikidata_entity_id,
                    },
                    "is_nil": False,
                },
                "text": span.text,
                "title": entity.wikipedia_entity_title,
                "url": "https://www.wikidata.org/wiki/" + entity.wikidata_entity_id,
            }
            if entity and entity.wikidata_entity_id and entity.wikipedia_entity_title
            else {"text": span.text, "linking": {"is_nil": True}}
        )
        # Ensure type is never None/null when writing to JSON
        span_type = span.coarse_mention_type or "UNKNOWN"
        base_obj = {
           "text": span.text, "start" : span.start, "end" : span.start+ span.ln, "type": span_type, "id": index 
        }
        if features_object is not None: 
            base_obj['features'] = features_object
        return base_obj
    def _is_valid_date(date_time: str):
        return pd.to_datetime(date_time, errors='coerce') is not pd.NaT
    def _convert_to_gate(self, text, entities):
        return {
            "text": text,
            "annotation_sets": {
                "entities_": {
                    "annotations": entities
                }
            }
        }
    def process(self, docs : Dict[str, Any] , save: bool = False) -> Dict[str, Any]:
        processed: Dict[str, Any] = {}
        out_dir = Path("output")                # top-level folder you want
        out_dir.mkdir(parents=True, exist_ok=True)
        for key, value in tqdm(docs.items()):
            spans = [
               self._create_span(span, index)
                for index, span in enumerate(self.model.process_text(value))
                if len(span.text) > 2
            ]
            if save:
                base = self._convert_to_gate(value, spans)
                clusters = self.cluster_mentions(base)
                base['features']  = {"clusters": {"entities_": clusters} }
                base['name'] = key
                base['preview'] = value[:100] + ' ...' if len(value) > 100 else value + '...'
                p = Path(key)  # preserves subfolders if present
                target = out_dir / p
                target = target.with_suffix(".json")  # replace/add .json extension
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open("w", encoding="utf-8") as f:
                    json.dump(base, f, ensure_ascii=False)
            else:
                processed[key] = self._convert_to_gate(value, spans)
                clusters = self.cluster_mentions(processed[key])
                processed[key]['features']  = {"clusters": {"entities_": clusters} }
                processed[key]['name'] = key
                processed[key]['preview'] = value[:100] + ' ...' if len(value) > 100 else value + '...'

        return processed

    def cluster_mentions(self, gate_doc: Dict[str, Any], text_similarity_threshold: float = 0.8):
        """Cluster mentions in a GATE-style document using Wikidata ids and text similarity.

        Returns clusters with shape:
        {id, title, type, nelements, mentions: [{id, mention}, ...]}
        """
        annotations = gate_doc.get("annotation_sets", {}).get("entities_", {}).get("annotations", [])

        clusters = []
        used_indices = set()

        # Helper: get mention text and mention id
        def _mention_info(idx, ann):
            mention_text = ann.get("features", {}).get("text") or ann.get("text") or ""
            mention_id = ann.get("id", idx)
            return mention_id, mention_text

        # 1) Group by Wikidata id (if available)
        id_map = {}
        for i, ann in enumerate(annotations):
            url = ann.get("features", {}).get("linking", {}).get("top_candidate", {}).get("url")
            wid = None
            if url and isinstance(url, str):
                wid = url.split("/")[-1]
            # fallback to explicit wikidata_id feature if present
            if not wid:
                wid = ann.get("features", {}).get("wikidata_id")
            if wid:
                id_map.setdefault(wid, []).append(i)

        cluster_id = 1
        for wid, idxs in id_map.items():
            mentions = []
            for idx in idxs:
                ann = annotations[idx]
                mid, mtext = _mention_info(idx, ann)
                mentions.append({"id": mid, "mention": mtext})
                used_indices.add(idx)

            # title: prefer wiki title from linking, else use wikidata id
            first_ann = annotations[idxs[0]]
            title = first_ann.get("features", {}).get("linking", {}).get("title") or wid
            ent_type = first_ann.get("type") or first_ann.get("features", {}).get("entity_type") or "ENTITY"

            clusters.append({
                "id": cluster_id,
                "title": title,
                "type": ent_type,
                "nelements": len(mentions),
                "mentions": mentions,
            })
            cluster_id += 1

        # 2) Cluster remaining annotations by text similarity
        for i, ann in enumerate(annotations):
            if i in used_indices:
                continue
            mid, mtext = _mention_info(i, ann)
            placed = False
            for cluster in clusters:
                # representative mention is first in cluster
                rep_text = cluster["mentions"][0]["mention"] if cluster.get("mentions") else ""
                ratio = difflib.SequenceMatcher(None, mtext.lower(), rep_text.lower()).ratio()
                if ratio >= text_similarity_threshold:
                    cluster["mentions"].append({"id": mid, "mention": mtext})
                    cluster["nelements"] = cluster.get("nelements", 0) + 1
                    placed = True
                    break
            if not placed:
                ent_type = ann.get("type") or ann.get("features", {}).get("entity_type") or "ENTITY"
                clusters.append({
                    "id": cluster_id,
                    "title": mtext,
                    "type": ent_type,
                    "nelements": 1,
                    "mentions": [{"id": mid, "mention": mtext}],
                })
                cluster_id += 1

        return clusters

In [4]:
def read_docs(path: str, limit: int = -1) -> Dict[str,str]:
    texts: Dict[str, str] = {}
    p = Path(path)
    files = sorted(p.glob("*.txt"))
    for index, doc in enumerate(files):
        if limit != -1 and index >= limit:
            break
        if not doc.is_file():
            continue
        try:
            with doc.open("r", encoding="utf-8", errors="replace") as f:
                texts[doc.name] = f.read()
        except Exception as e:
            print(f"error processing {doc.name}: {e}")
    return texts

In [19]:
extractor = PipelineInformationExtractorRefined()

In [6]:
docs = read_docs("./testDocs/out")

In [20]:
result = extractor.process(docs, True)

  0%|          | 0/7641 [00:00<?, ?it/s]

In [ ]:
result["0035224228.txt"]

In [8]:
from pathlib import Path
import json

out_dir = Path("output")                # top-level folder you want
out_dir.mkdir(parents=True, exist_ok=True)

for key, value in result.items():
    p = Path(key)  # preserves subfolders if present
    target = out_dir / p
    target = target.with_suffix(".json")  # replace/add .json extension
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("w", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False)